# Notebook 6: Hard-Pair Correction from z

This notebook tests whether `z` contains actionable information for the backbone's hardest class confusions. It compares `phase_b.pt` and `phase_c.pt` side by side and never retrains.


In [ ]:
# Colab-first setup: clone/update the repo, install it, and mount Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Running outside Colab; using the current working tree.")


In [ ]:
from flow_circuits.data import CIFAR10_STATS, build_cifar10_splits
from flow_circuits.evaluation import (
    NB06_EXPERIMENT_IDS,
    run_hard_pair_case_study_experiment,
    run_hard_pair_hybrid_correction_experiment,
    run_hard_pair_probe_benchmark_experiment,
    run_multiclass_z_probe_audit_experiment,
)
from flow_circuits.training import collect_interpretability_outputs, load_components_from_checkpoint


## Config

Change the single `EXPERIMENTS` line below to run all four `nb06` experiments or any subset.


In [ ]:
from pathlib import Path

RUN_MODE = "full"  # "debug" or "full"
CONFIG_NAME = "resnet18_aligned"
EXPERIMENTS = "all"
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits")
EXPERIMENT_ROOT = DRIVE_ROOT / "experiments" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb06_hard_pair_correction_from_z" / CONFIG_NAME

PHASE_B_CHECKPOINT = EXPERIMENT_ROOT / "phase_b.pt"
PHASE_C_CHECKPOINT = EXPERIMENT_ROOT / "phase_c.pt"


In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch

try:
    import pandas as pd
except ImportError:
    pd = None

MODEL_ORDER = [("phase_b", "Phase B"), ("phase_c", "Phase C")]
EXPERIMENT_LABELS = {
    "multiclass_probe_audit": "Multiclass Probe Audit",
    "hard_pair_probe_benchmark": "Hard-Pair Probe Benchmark",
    "hybrid_correction": "Hybrid Correction",
    "correction_case_studies": "Correction Case Studies",
}

AUTO_N_JOBS = max(1, min(8, (os.cpu_count() or 1) - 1 if (os.cpu_count() or 1) > 1 else 1))

if EXPERIMENTS == "all":
    SELECTED_EXPERIMENTS = list(NB06_EXPERIMENT_IDS)
else:
    SELECTED_EXPERIMENTS = [str(name) for name in EXPERIMENTS]
invalid = sorted(set(SELECTED_EXPERIMENTS) - set(NB06_EXPERIMENT_IDS))
if invalid:
    raise ValueError(f"Unknown experiments: {invalid}. Valid ids: {NB06_EXPERIMENT_IDS}")

if not PHASE_B_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_B_CHECKPOINT}")
if not PHASE_C_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing required checkpoint: {PHASE_C_CHECKPOINT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR = OUTPUT_DIR / "comparisons"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
components_by_tag = {
    tag: load_components_from_checkpoint(path, device=device)
    for tag, path in (("phase_b", PHASE_B_CHECKPOINT), ("phase_c", PHASE_C_CHECKPOINT))
}

base_config = components_by_tag["phase_b"].config
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=base_config["data"]["batch_size"],
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)

SETTINGS_BY_MODE = {
    "debug": {
        "multiclass_probe_audit": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "n_jobs": AUTO_N_JOBS},
        "hard_pair_probe_benchmark": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "top_k_nodes": 3, "n_jobs": AUTO_N_JOBS},
        "hybrid_correction": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "top_k_nodes": 3, "n_jobs": AUTO_N_JOBS},
        "correction_case_studies": {"fit_max_images": 1000, "val_max_images": 500, "test_max_images": 500, "top_pairs": 3, "top_k_nodes": 3, "exemplar_count": 6, "n_jobs": AUTO_N_JOBS},
    },
    "full": {
        "multiclass_probe_audit": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "n_jobs": AUTO_N_JOBS},
        "hard_pair_probe_benchmark": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "top_k_nodes": 5, "n_jobs": AUTO_N_JOBS},
        "hybrid_correction": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "top_k_nodes": 5, "n_jobs": AUTO_N_JOBS},
        "correction_case_studies": {"fit_max_images": 4000, "val_max_images": 1000, "test_max_images": 1000, "top_pairs": 5, "top_k_nodes": 5, "exemplar_count": 6, "n_jobs": AUTO_N_JOBS},
    },
}
if RUN_MODE not in SETTINGS_BY_MODE:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

results_dir_by_tag = {tag: OUTPUT_DIR / tag for tag, _label in MODEL_ORDER}
for path in results_dir_by_tag.values():
    path.mkdir(parents=True, exist_ok=True)

RESULTS = {}
_OUTPUT_CACHE = {}


def _should_run(experiment_id):
    return experiment_id in SELECTED_EXPERIMENTS


def _cache_path(tag, experiment_id):
    return results_dir_by_tag[tag] / f"{experiment_id}.json"


def _format_seconds(seconds):
    seconds = max(int(seconds), 0)
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def _progress_logger(**event):
    stamp = time.strftime("%H:%M:%S")
    label = "Phase B" if event.get("checkpoint_tag") == "phase_b" else "Phase C"
    experiment = event.get("experiment")
    stage = str(event.get("stage", "")).replace("_", " ")
    completed = event.get("completed")
    total = event.get("total")
    elapsed = _format_seconds(event.get("elapsed_seconds", 0.0))
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    if total:
        prefix = f"{stage} {completed}/{total}"
        eta_text = f" | ETA {_format_seconds(eta)}" if eta is not None else ""
    else:
        prefix = stage
        eta_text = ""
    print(f"[{stamp}] {label} | {experiment} | {prefix} | elapsed {elapsed}{eta_text} | {message}")


def _run_or_load(tag, experiment_id, runner):
    cache_path = _cache_path(tag, experiment_id)
    if cache_path.exists() and not FORCE_RERUN:
        print(f"Loading cached {experiment_id} for {tag}: {cache_path}")
        return json.loads(cache_path.read_text(encoding="utf-8"))
    print(f"Running {experiment_id} for {tag} ...")
    return runner(cache_path)


def _write_summary_table(rows, title):
    print(title)
    if not rows:
        print("(no rows)")
        return
    if pd is None:
        for row in rows:
            print(row)
        return
    display(pd.DataFrame(rows))


def _denormalize_image(image_tensor):
    mean = torch.tensor(CIFAR10_STATS["mean"], dtype=image_tensor.dtype).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STATS["std"], dtype=image_tensor.dtype).view(3, 1, 1)
    image = image_tensor.cpu() * std + mean
    return image.clamp(0.0, 1.0).permute(1, 2, 0).numpy()


def _draw_overlay(ax, overlay):
    if not overlay:
        return
    for box in overlay.get("active_boxes", []):
        color = "red" if box.get("is_representative") else "cyan"
        rect = patches.Rectangle((box["x0"], box["y0"]), box["x1"] - box["x0"], box["y1"] - box["y0"], linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)


def _get_outputs(tag, split_name, max_images):
    key = (tag, split_name, int(max_images))
    if key not in _OUTPUT_CACHE:
        _OUTPUT_CACHE[key] = collect_interpretability_outputs(
            components_by_tag[tag],
            loaders[split_name],
            device=device,
            max_images=max_images,
        )
    return _OUTPUT_CACHE[key]


def _show_examples(example_rows, outputs, title):
    if not example_rows:
        print(f"{title}: no examples")
        return
    ncols = min(3, len(example_rows))
    nrows = int(np.ceil(len(example_rows) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)
    for ax in axes.flatten():
        ax.axis("off")
    for ax, row in zip(axes.flatten(), example_rows):
        image = _denormalize_image(outputs["images"][row["row_index"]])
        ax.imshow(image)
        _draw_overlay(ax, row.get("overlay"))
        ax.set_title(f"idx={row['dataset_index']} | label={row.get('true_label_name', row.get('label_name', '?'))}")
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def _show_node_heatmap(rows, value_key, title):
    if not rows:
        print(f"{title}: no rows")
        return
    n_layers = 1 + max(int(row["layer_idx"]) for row in rows)
    n_cells = 1 + max(int(row["cell_idx"]) for row in rows)
    grid = np.full((n_layers, n_cells), np.nan, dtype=float)
    for row in rows:
        grid[int(row["layer_idx"]), int(row["cell_idx"])] = float(row[value_key])
    fig, ax = plt.subplots(figsize=(max(6, n_cells / 2), max(3, n_layers / 1.5)))
    im = ax.imshow(grid, aspect="auto", cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("cell_idx")
    ax.set_ylabel("layer_idx")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


## Experiment 1: Multiclass Probe Audit

Measure how much class information is linearly decodable from `z` globally, by layer, and by node.


In [ ]:
if _should_run("multiclass_probe_audit"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["multiclass_probe_audit"]
    RESULTS["multiclass_probe_audit"] = {
        tag: _run_or_load(
            tag,
            "multiclass_probe_audit",
            lambda cache_path, tag=tag: run_multiclass_z_probe_audit_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                val_max_images=settings["val_max_images"],
                test_max_images=settings["test_max_images"],
                n_jobs=settings["n_jobs"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping multiclass_probe_audit.")


In [ ]:
if "multiclass_probe_audit" in RESULTS:
    _write_summary_table([
        {"checkpoint": label, **RESULTS["multiclass_probe_audit"][tag]["summary"]}
        for tag, label in MODEL_ORDER
    ], title="Multiclass probe audit summary")
    for tag, label in MODEL_ORDER:
        _write_summary_table(RESULTS["multiclass_probe_audit"][tag]["top_nodes_by_accuracy"], title=f"{label} top nodes by held-out accuracy")
        _show_node_heatmap(RESULTS["multiclass_probe_audit"][tag]["per_node"], "accuracy", f"{label} per-node accuracy heatmap")


## Experiment 2: Hard-Pair Probe Benchmark

Select hard pairs from validation confusion and compare backbone, full-`z`, and top-node probes on test.


In [ ]:
if _should_run("hard_pair_probe_benchmark"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["hard_pair_probe_benchmark"]
    RESULTS["hard_pair_probe_benchmark"] = {
        tag: _run_or_load(
            tag,
            "hard_pair_probe_benchmark",
            lambda cache_path, tag=tag: run_hard_pair_probe_benchmark_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                val_max_images=settings["val_max_images"],
                test_max_images=settings["test_max_images"],
                top_pairs=settings["top_pairs"],
                top_k_nodes=settings["top_k_nodes"],
                n_jobs=settings["n_jobs"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping hard_pair_probe_benchmark.")


In [ ]:
if "hard_pair_probe_benchmark" in RESULTS:
    _write_summary_table([
        {"checkpoint": label, **RESULTS["hard_pair_probe_benchmark"][tag]["summary"]}
        for tag, label in MODEL_ORDER
    ], title="Hard-pair benchmark summary")
    for tag, label in MODEL_ORDER:
        rows = RESULTS["hard_pair_probe_benchmark"][tag]["pair_rows"]
        _write_summary_table(rows, title=f"{label} hard-pair results")
        if rows:
            _write_summary_table(rows[0]["selected_top_nodes"], title=f"{label} top nodes for hardest pair")


## Experiment 3: Hybrid Correction

Use pairwise `z` probes as top-2 hard-pair tie-breakers and compare against the backbone on both the trigger subset and the full test set.


In [ ]:
if _should_run("hybrid_correction"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["hybrid_correction"]
    RESULTS["hybrid_correction"] = {
        tag: _run_or_load(
            tag,
            "hybrid_correction",
            lambda cache_path, tag=tag: run_hard_pair_hybrid_correction_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                val_max_images=settings["val_max_images"],
                test_max_images=settings["test_max_images"],
                top_pairs=settings["top_pairs"],
                top_k_nodes=settings["top_k_nodes"],
                n_jobs=settings["n_jobs"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping hybrid_correction.")


In [ ]:
if "hybrid_correction" in RESULTS:
    _write_summary_table([
        {
            "checkpoint": label,
            **RESULTS["hybrid_correction"][tag]["summary"],
        }
        for tag, label in MODEL_ORDER
    ], title="Hybrid correction summary")
    for tag, label in MODEL_ORDER:
        _write_summary_table(RESULTS["hybrid_correction"][tag]["full_z_hybrid"]["per_pair_rows"], title=f"{label} full-z hybrid per-pair results")
        _write_summary_table(RESULTS["hybrid_correction"][tag]["top_node_hybrid"]["per_pair_rows"], title=f"{label} top-node hybrid per-pair results")


## Experiment 4: Correction Case Studies

Render corrected, harmed, and unchanged trigger examples with top-node overlays.


In [ ]:
if _should_run("correction_case_studies"):
    settings = SETTINGS_BY_MODE[RUN_MODE]["correction_case_studies"]
    RESULTS["correction_case_studies"] = {
        tag: _run_or_load(
            tag,
            "correction_case_studies",
            lambda cache_path, tag=tag: run_hard_pair_case_study_experiment(
                components_by_tag[tag],
                loaders["fit"],
                loaders["val"],
                loaders["test"],
                device=device,
                checkpoint_tag=tag,
                fit_max_images=settings["fit_max_images"],
                val_max_images=settings["val_max_images"],
                test_max_images=settings["test_max_images"],
                top_pairs=settings["top_pairs"],
                top_k_nodes=settings["top_k_nodes"],
                exemplar_count=settings["exemplar_count"],
                n_jobs=settings["n_jobs"],
                output_path=cache_path,
                progress_callback=_progress_logger,
            ),
        )
        for tag, _label in MODEL_ORDER
    }
else:
    print("Skipping correction_case_studies.")


In [ ]:
if "correction_case_studies" in RESULTS:
    _write_summary_table([
        {"checkpoint": label, **RESULTS["correction_case_studies"][tag]["summary"]}
        for tag, label in MODEL_ORDER
    ], title="Correction case study summary")
    settings = SETTINGS_BY_MODE[RUN_MODE]["correction_case_studies"]
    for tag, label in MODEL_ORDER:
        outputs = _get_outputs(tag, "test", settings["test_max_images"])
        for pair_row in RESULTS["correction_case_studies"][tag]["pair_case_studies"][:2]:
            _show_examples(pair_row["corrected_examples"], outputs, f"{label} corrected examples: {pair_row['trigger_pair_name']}")
            _show_examples(pair_row["harmed_examples"], outputs, f"{label} harmed examples: {pair_row['trigger_pair_name']}")


## Final Summary


In [ ]:
comparison_rows = []
if "multiclass_probe_audit" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["multiclass_probe_audit"], "phase_b": RESULTS["multiclass_probe_audit"]["phase_b"]["summary"]["global_accuracy"], "phase_c": RESULTS["multiclass_probe_audit"]["phase_c"]["summary"]["global_accuracy"], "metric": "global_accuracy"})
if "hard_pair_probe_benchmark" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["hard_pair_probe_benchmark"], "phase_b": RESULTS["hard_pair_probe_benchmark"]["phase_b"]["summary"]["mean_full_z_probe_accuracy"], "phase_c": RESULTS["hard_pair_probe_benchmark"]["phase_c"]["summary"]["mean_full_z_probe_accuracy"], "metric": "mean_full_z_probe_accuracy"})
if "hybrid_correction" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["hybrid_correction"], "phase_b": RESULTS["hybrid_correction"]["phase_b"]["summary"]["top_node_hybrid_overall_accuracy"], "phase_c": RESULTS["hybrid_correction"]["phase_c"]["summary"]["top_node_hybrid_overall_accuracy"], "metric": "top_node_hybrid_overall_accuracy"})
if "correction_case_studies" in RESULTS:
    comparison_rows.append({"experiment": EXPERIMENT_LABELS["correction_case_studies"], "phase_b": RESULTS["correction_case_studies"]["phase_b"]["summary"]["n_corrected_examples"], "phase_c": RESULTS["correction_case_studies"]["phase_c"]["summary"]["n_corrected_examples"], "metric": "n_corrected_examples"})
_write_summary_table(comparison_rows, title="Hard-pair correction summary")
(COMPARISON_DIR / "summary.json").write_text(json.dumps(comparison_rows, indent=2), encoding="utf-8")
